In [ ]:
import re
from pathlib import Path
import pandas as pd


xlsx_path = Path(r"C:/Users/De/pathogens 2019-2024.xlsx")

BACT_COL = "Microorganisms"
COUNT_COL = "Total"

TAIL_DROP = re.compile(r"\b(ESBL|MRS|MRSA)\b", flags=re.IGNORECASE)

GROUP_DROP = re.compile(r"\s*\((Group\s*[A-Z]|group\s*[a-z])\)\s*", flags=re.IGNORECASE)

SUBSP_DROP = re.compile(r"\bsubsp\..*$", flags=re.IGNORECASE)

def basic_clean(name: str) -> str:
    if pd.isna(name):
        return name
    s = str(name).strip()
    s = re.sub(r"\s+", " ", s)

      s = re.sub(r"\bmethicillin\s+resistant\b", "", s, flags=re.IGNORECASE).strip()

    s = TAIL_DROP.sub("", s)
    s = GROUP_DROP.sub("", s)
    s = SUBSP_DROP.sub("", s)

    s = re.sub(r"\s+", " ", s).strip()
    return s

def group_bacteria(name: str) -> str:
    if pd.isna(name):
        return name

    s = name

    # ---- Staphylococcus aureus (including Methicillin Resistant Staphylococcus aureus) ----
       if re.search(r"methicillin\s+resistant\s+staphylococcus\s+aureus", s, re.IGNORECASE):
        return "Staphylococcus aureus"
    if re.search(r"staphylococcus\s+aureus", s, re.IGNORECASE):
        return "Staphylococcus aureus"

    # ---- Staphylococcus spp. ----
    if s.lower().startswith("staphylococcus"):
        if re.search(r"staphylococcus\s+saprophyticus", s, re.IGNORECASE):
            return "Staphylococcus saprophyticus"
        return "CoNS" 
        
    # ---- Streptococcus ----
    if s.lower().startswith("streptococcus"):
        if re.search(r"streptococcus\s+pyogenes", s, re.IGNORECASE):
            return "Streptococcus pyogenes"
        if re.search(r"streptococcus\s+pneumoniae", s, re.IGNORECASE):
            return "Streptococcus pneumoniae"
        if re.search(r"streptococcus\s+agalactiae", s, re.IGNORECASE):
            return "Streptococcus agalactiae"
        return "Streptococcus spp."

    return s

xls = pd.ExcelFile(xlsx_path)

year_sheets = []
for sh in xls.sheet_names:
    m = re.fullmatch(r"\s*(2019|2020|2021|2022|2023|2024)\s*", str(sh))
    if m:
        year_sheets.append((int(m.group(1)), sh))
year_sheets = sorted(year_sheets)

if not year_sheets:
    raise ValueError("not found 2019–2024")

all_data = []

for year, sh in year_sheets:
    df = pd.read_excel(xlsx_path, sheet_name=sh).dropna(how="all")

    if BACT_COL not in df.columns or COUNT_COL not in df.columns:
        raise KeyError(f"list {sh}: Required columns are missing '{BACT_COL}' and/or '{COUNT_COL}'")

    tmp = df[[BACT_COL, COUNT_COL]].copy()
    tmp.columns = ["bacteria_raw", "count"]

    tmp["count"] = pd.to_numeric(tmp["count"], errors="coerce")
    tmp = tmp.dropna(subset=["count"])
    tmp = tmp[tmp["count"] > 0]

    tmp["year"] = year

    # cleaning + grouping
    tmp["bacteria_clean"] = tmp["bacteria_raw"].apply(basic_clean)
    tmp["bacteria_group"] = tmp["bacteria_clean"].apply(group_bacteria)

    tmp = tmp.groupby(["year", "bacteria_group"], as_index=False)["count"].sum()
    all_data.append(tmp)

counts_all = pd.concat(all_data, ignore_index=True)


year_totals = (
    counts_all.groupby("year", as_index=False)["count"]
    .sum()
    .rename(columns={"count": "year_total"})
)

counts_all = counts_all.merge(year_totals, on="year", how="left")
counts_all["share"] = counts_all["count"] / counts_all["year_total"]
counts_all["share_pct"] = (counts_all["share"] * 100).round(2)


counts_all["rank"] = counts_all.groupby("year")["count"].rank(method="first", ascending=False)

top10 = (
    counts_all[counts_all["rank"] <= 10]
    .sort_values(["year", "rank"])
    .copy()
)

display(top10[["year", "bacteria_group", "count", "share_pct", "rank"]])


out_path = xlsx_path.parent / "top10_pathogens_2019_2024_grouped_counts_shares.xlsx"
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    counts_all.to_excel(writer, sheet_name="All_grouped", index=False)
    top10.to_excel(writer, sheet_name="Top10_by_year", index=False)

print(f"Done. Results have been saved: {out_path}")


In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


base_dir = Path(r"C:/Users/De")
in_path = base_dir / "top10_pathogens_2019_2024_grouped_counts_shares.xlsx"

if not in_path.exists():
    raise FileNotFoundError(f"Results file not found: {in_path}")

top10 = pd.read_excel(in_path, sheet_name="Top10_by_year")

req_cols = {"year", "bacteria_group", "count", "share_pct", "rank"}
missing = req_cols - set(top10.columns)
if missing:
    raise KeyError(f"В Top10_by_year Required columns are missing: {missing}. Found: {list(top10.columns)}")

top10["year"] = top10["year"].astype(int)
top10 = top10.sort_values(["year", "rank"]).copy()
years = sorted(top10["year"].unique())

def short_label(name: str) -> str:
    if pd.isna(name):
        return name
    s = str(name).strip()

    if s in {"CoNS", "Streptococcus spp.", "Achromobacter spp.", "Corynebacterium spp.", "Cedecea spp."}:
        return s
    if s.endswith(" spp."):
        return s

    parts = s.split()
    if len(parts) >= 2:
        return f"{parts[0][0]}. {parts[1]}"
    return s

top10["label_short"] = top10["bacteria_group"].apply(short_label)

BRIGHT_COLOUR = [
    "#7B5CD6",  
    "#FF7F7F",  
    "#BFBFBF",  
    "#9E9E9E",  
    "#B39DDB",  
    "#6D6D6D",  
    "#E6D74A",  
    "#C97A63",  
    "#4FC3D9", 
    "#8BC34A",  

]

all_taxa = sorted(top10["bacteria_group"].unique())
taxon_to_color = {
    tax: BRIGHT_COLOUR[i % len(BRIGHT_COLOUR)]
    for i, tax in enumerate(all_taxa)
}

XMAX = 50  

def style_axes(ax):
    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.grid(axis="x", linestyle="--", linewidth=1, alpha=0.6)
    ax.set_axisbelow(True)
    ax.set_xlim(0, XMAX)


def plot_top10_year(df_year: pd.DataFrame, year: int, out_png: Path, out_pdf: Path) -> None:
    df = df_year.sort_values("share_pct", ascending=True).copy()

    y_labels = df["label_short"].tolist()
    x_vals = df["share_pct"].to_numpy()
    colors = [taxon_to_color[t] for t in df["bacteria_group"]]

    fig_h = max(6, 0.55 * len(df) + 1.2)
    fig, ax = plt.subplots(figsize=(12, fig_h))

    bars = ax.barh(y_labels, x_vals, color=colors)

    for b, v in zip(bars, x_vals):
        ax.text(
            min(v, XMAX) + 0.6,
            b.get_y() + b.get_height() / 2,
            f"{v:.1f}%",
            va="center",
            ha="left",
            fontsize=12,
        )

    ax.set_title(f"{year}", fontsize=22, pad=14)
    ax.set_xlabel("Share, %", fontsize=14)

    style_axes(ax)
    plt.tight_layout()

    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    fig.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)


out_dir = base_dir / "top10_figures_2019_2024"
out_dir.mkdir(parents=True, exist_ok=True)

for y in years:
    df_y = top10[top10["year"] == y].copy()
    plot_top10_year(
        df_y,
        y,
        out_dir / f"top10_isolates_{y}.png",
        out_dir / f"top10_isolates_{y}.pdf",
    )

print(f"Done: separate figures are saved {out_dir}")


ncols = 3
nrows = int(np.ceil(len(years) / ncols))

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(18, 14))
axes = np.array(axes).reshape(-1)

for i, y in enumerate(years):
    ax = axes[i]
    df_y = top10[top10["year"] == y].sort_values("share_pct", ascending=True)

    y_labels = df_y["label_short"].tolist()
    x_vals = df_y["share_pct"].to_numpy()
    colors = [taxon_to_color[t] for t in df_y["bacteria_group"]]

    bars = ax.barh(y_labels, x_vals, color=colors)

    for b, v in zip(bars, x_vals):
        ax.text(
            min(v, XMAX) + 0.4,
            b.get_y() + b.get_height() / 2,
            f"{v:.1f}%",
            va="center",
            ha="left",
            fontsize=9,
        )

    ax.set_title(f"{y}", fontsize=14, pad=8)
    ax.set_xlabel("Share, %", fontsize=11)
    style_axes(ax)

for j in range(len(years), len(axes)):
    axes[j].axis("off")

plt.tight_layout()

fig.savefig(out_dir / "top10_isolates_2019_2024_ALL.png", dpi=300, bbox_inches="tight")
fig.savefig(out_dir / "top10_isolates_2019_2024_ALL.pdf", bbox_inches="tight")
plt.close(fig)

print("Done: The main file has been saved")